# 📝 과제 01: 온라인 쇼핑 상담 API 만들기

<div class="day77-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">실행:</strong> RunPod 파드 하나에서 vLLM 서버(터미널 1), 과제 FastAPI 서버(터미널 2), 이 노트북을 모두 실행합니다. <strong style="font-size:1em;color:#1f3a5f;">내 PC에서 실행하는 코드는 없습니다.</strong><br><strong style="font-size:1em;color:#1f3a5f;">작성:</strong> 서버 코드는 이 과제 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code> 패키지 .py 파일에 씁니다. 노트북은 실행 중인 서버(127.0.0.1:8080)에 요청을 보내 자가채점만 합니다.<br><strong style="font-size:1em;color:#1f3a5f;">순서:</strong> 과제 02 폴더에는 이 과제의 완성 코드가 들어 있으니, 이 과제를 마친 뒤에 엽니다.<br><strong style="font-size:1em;color:#1f3a5f;">준비:</strong> 교안 01을 마친 상태에서 시작합니다. 교안 01의 vLLM 서버와 원격 저장소의 .env(VLLM_API_KEY)를 그대로 씁니다.</div>

## 만들 결과

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">교안 01에서 만든 노무 상담 서빙 API를 본떠 온라인 쇼핑 상담 창구(전자상거래법 청약철회, 환급)를 만듭니다. 앱에 주소 묶음을 붙이고 쇼핑 질문의 요청 양식을 정한 다음, 답 전체를 돌려주는 주소와 조각을 스트리밍하는 주소를 차례로 만듭니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/main.py</code>: 제목이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;온라인 쇼핑 상담 API&quot;</code>인 앱 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">app</code>과 주소 묶음 두 개(상태 확인, 쇼핑 상담)</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/schemas/chat.py</code>: 질문(2자 이상 300자 이하), 생각 켜기, 주문번호(선택)를 받는 요청 양식 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code></li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/routers/shop_router.py</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/shop</code> 주소 묶음 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_router</code>, 사용자 메시지 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">make_user_message</code>, 주소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat/stream</code></li>
</ul>
</div>

## 준비: 서버와 제공 코드

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code>는 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code>과 같은 계층 폴더입니다(교안 01 2-1).</p>
<table style="background:#ffffff;color:#202124;border-collapse:collapse;margin:8px 0;font-size:16px;line-height:1.6;">
<thead>
<tr>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">폴더, 파일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">하는 일</th>
  <th style="background:#edf2f7;color:#1f3a5f;border:1px solid #cbd5e1;padding:6px 10px;text-align:left;vertical-align:top;line-height:1.6;">이 과제에서</th>
</tr>
</thead>
<tbody>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">main.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">앱 조립: 앱을 만들고 주소 묶음을 붙입니다.</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">1번 작성</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">schemas/chat.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">요청과 응답의 양식</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">2번 작성(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code>), 나머지는 제공</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/shop_router.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">쇼핑 상담 주소 묶음</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">3번 작성</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/system_router.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">상태 확인 주소 묶음</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">제공</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/backend.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/streaming.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">vLLM 요청, 조각을 SSE로 중계</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">제공</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">환경 변수와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에서 읽는 설정</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">제공</td>
</tr>
<tr>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">resources/</code></td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">시스템 메시지에 붙일 참고 조문</td>
  <td style="border:1px solid #cbd5e1;padding:6px 10px;vertical-align:top;line-height:1.6;">제공</td>
</tr>
</tbody>
</table>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">환경:</strong> VS Code Remote SSH로 연 RunPod 원격 창의 노트북과 터미널을 씁니다. 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code> 커널을 고릅니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 참고 조문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/resources/commerce_reference.json</code>(전자상거래법 청약철회, 환급·반환 비용)이고, 서버가 시스템 메시지에 붙입니다. 자가채점에 쓰는 질문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/과제/commerce_questions.jsonl</code>의 첫 질문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">제공 코드:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">core/config.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/backend.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">services/streaming.py</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/system_router.py</code>는 교안 01의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app</code>과 같은 코드라서, 함수 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">make_system_prompt</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_vllm</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">open_vllm_stream</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">to_chat_response</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_stream</code>, 응답 양식 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ChatResponse</code>, 상태 확인 주소 묶음 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router</code>를 그대로 씁니다. 생각을 켠 요청은 설정의 생성 상한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens_think</code>(4400)와 모델 카드 권장 샘플링값(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think_</code>로 시작하는 설정)을 씁니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">max_tokens_think_labor</code>는 교안 01 노무 창구용이라 이 과제에서는 쓰지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 01과 다른 점:</strong><ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">주소는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/shop</code>으로 시작합니다.</li>
<li style="margin:3px 0;">질문은 300자까지 받고, 주문번호 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">order_id</code>를 함께 받을 수 있습니다.</li>
<li style="margin:3px 0;">지침 문장은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다.&quot;</code>이고 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">routers/shop_router.py</code>에 들어 있습니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성:</strong><ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code>의 .py 파일에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code> 주석 아래를 채웁니다. 문항의 요구사항 (n)은 그 파일의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code>에 씁니다.</li>
<li style="margin:3px 0;">파일을 저장하면 터미널 2의 서버가 저절로 다시 뜨므로, 저장한 뒤 그 문항의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[자가채점]</code> 셀을 실행합니다. 1번부터 차례로 풉니다.</li>
</ul>
</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">서버가 응답하지 않을 때:</strong> 자가채점 셀이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ConnectError</code>나 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ReadTimeout</code>으로 멈추면 터미널 2의 서버가 떠 있지 않거나 코드를 읽다가 오류가 난 것입니다. 터미널 2의 마지막 오류 줄에서 파일 이름과 줄 번호를 확인합니다.</li>
</ul>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 주소 ----
import json
from pathlib import Path

import httpx

lesson_dir = Path(".").resolve()  # 이 노트북의 단원 폴더(정답 노트북은 "..")
day_dir = lesson_dir.parent                # data, images 가 있는 일차 폴더
data_dir = day_dir / "data"

SERVING_URL = "http://127.0.0.1:8080"   # 터미널 2에서 띄운 FastAPI 서빙 API
SERVED_NAME = "qwen3.5-4b"            # vLLM의 --served-model-name

In [ ]:
def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(서빙 벤치마킹 단원과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# 자가채점에 쓸 쇼핑 질문(첫 질문 하나)
commerce_questions = load_jsonl(data_dir / "과제" / "commerce_questions.jsonl")
sample_question = commerce_questions[0]["question"]

# 서버 응답을 기다리는 최대 시간(초). 서버의 vLLM 시간 제한(기본 120초)보다 길게 둡니다
REQUEST_TIMEOUT = 180

# 이 질문 앞에 "[주문번호 ORD-123456] " 을 붙이면 늘어나는 입력 토큰 수(Qwen3.5-4B 토크나이저로 센 값)
ORDER_PREFIX_TOKENS = 11
print("질문 수:", len(commerce_questions), "| 자가채점 질문:", sample_question)

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">vLLM 서버(터미널 1):</strong> 교안 01의 vLLM 서버가 떠 있으면 그대로 씁니다. 서버가 없을 때만 RunPod 터미널 1에서 아래 두 블록을 차례로 실행합니다. 첫 블록은 교안 01에서 원격 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>에 저장한 vLLM 키를 불러옵니다(새 터미널에는 키가 없습니다).</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API"
pwd   # 끝이 day77_FastAPI_모델서빙API이면 맞습니다
mkdir -p output   # 서버 로그를 남길 폴더입니다
# 원격 저장소의 .env에 저장해 둔 vLLM 키를 불러옵니다(키를 새로 생성하지 않고, 화면에 찍지 않습니다)
VLLM_API_KEY="$(uv run python -c 'from dotenv import dotenv_values; print(dotenv_values("../.env")["VLLM_API_KEY"])')" &&
export VLLM_API_KEY
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 1</strong></div>

```bash
uv run vllm serve Qwen/Qwen3.5-4B \
  --served-model-name qwen3.5-4b \
  --max-model-len 8192 \
  --language-model-only \
  --reasoning-parser qwen3 \
  --default-chat-template-kwargs '{"enable_thinking": false}' \
  --enable-prompt-tokens-details \
  --enable-prefix-caching \
  --host 0.0.0.0 \
  --port 8000 \
  --api-key "$VLLM_API_KEY" 2>&1 | tee output/qwen3.5-4b_server.log
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">로그에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>가 나오면 vLLM 서버가 요청을 받을 수 있습니다. 서버 출력은 RunPod의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/qwen3.5-4b_server.log</code>에도 남습니다. <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>은 그대로 둡니다.</p>
<p style="margin:6px 0;">첫 실행은 모델 다운로드와 GPU 계산 준비에 몇 분이 걸릴 수 있습니다.</p>
<p style="margin:6px 0;">서버 로그 첫머리의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">non-default args</code> 줄에 vLLM 키가 찍히므로, 서버 로그나 터미널 1 화면은 다른 사람에게 공유하지 않습니다.</p>
</div>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">과제 서버(터미널 2):</strong> 교안 01의 서버(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving_app.main</code>)가 떠 있으면 터미널 2에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code>로 끕니다. 아래 두 블록으로 이 과제 폴더로 이동한 뒤 같은 포트 8080에 과제 서버를 띄웁니다. TODO를 채우기 전에도 서버는 빈 앱으로 뜹니다.</p>
</div>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
# 새 원격 터미널이면 이 블록부터 실행합니다
# 원격 저장소(실습자료) 맨 위 폴더를 찾아 그 안의 폴더로 이동합니다(저장소 안 어느 폴더에서 실행해도 됩니다)
cd "$(git rev-parse --show-toplevel)/day77_FastAPI_모델서빙API/과제_01_온라인쇼핑_상담_API_만들기"
pwd   # 끝이 과제_01_온라인쇼핑_상담_API_만들기이면 맞습니다
```

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · RunPod 터미널 2</strong></div>

```bash
uv run uvicorn shop_api.main:app --host 0.0.0.0 --port 8080 --reload
```

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">터미널 2에 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Application startup complete</code></strong>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Uvicorn running on http://0.0.0.0:8080</code>가 나오면 FastAPI 서버가 요청을 받을 수 있습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--reload</code>를 붙였으므로 서버 코드(.py)를 저장하면 서버가 저절로 다시 시작합니다.</p>
</div>

## 1. 앱과 라우팅

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> FastAPI 앱을 만들고 기능별 주소 묶음을 붙여, 쇼핑몰 화면과 상담 센터가 한 주소 체계로 상담 API를 쓰게 합니다. 상태 확인 주소는 서버가 요청을 받을 수 있는지 자동으로 점검할 때 부릅니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/main.py</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">app</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">FastAPI</code> 앱입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">title</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;온라인 쇼핑 상담 API&quot;</code>이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">description</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">version</code>도 줍니다(내용은 자유입니다). 제목은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code> 화면 맨 위와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">info.title</code>에 보입니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;">주소 묶음 붙이기:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">include_router</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">system_router</code>와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_router</code>를 붙입니다. 키 없이도 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /healthz</code>는 200과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;status&quot;: &quot;ok&quot;}</code>를, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GET /v1/models</code>는 200과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code>가 든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;models&quot;: [...]}</code>를 돌려줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_router</code>의 주소는 3번에서 만듭니다.</li>
</ul>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 01의 2-1(main.py)과 3절(라우팅)을 참고합니다.
세부구현
1) FastAPI 클래스로 앱을 만들 때 title, description, version 키워드 인자를 줍니다.
2) 앱의 include_router 메서드에 주소 묶음을 하나씩 넘깁니다. 묶음의 prefix가 주소 앞부분이 됩니다.
```

</details>

<div class="day77-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · RunPod 원격 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code 원격 창의 노트북. 내 PC가 아니라 RunPod에서 실행됩니다</span></div>

In [ ]:
# [자가채점] 앱 제목과 상태 확인 주소 두 개를 확인합니다
openapi = httpx.get(f"{SERVING_URL}/openapi.json", timeout=REQUEST_TIMEOUT).json()
assert openapi["info"]["title"] == "온라인 쇼핑 상담 API", f"앱 제목이 {openapi['info']['title']!r}입니다. main.py 의 TODO (1)에서 FastAPI 에 title=\"온라인 쇼핑 상담 API\"를 주세요. 터미널 2에서 이 과제 폴더의 과제 01 앱(main.py 의 app)을 실행했는지도 확인하세요."

# 상태 확인 주소: 키 없이 열리고 vLLM 을 부르지 않습니다
health = httpx.get(f"{SERVING_URL}/healthz", timeout=REQUEST_TIMEOUT)
assert health.status_code == 200, f"GET /healthz 가 {health.status_code}입니다. main.py 의 TODO (2)에서 include_router 로 system_router 를 붙였는지 확인하세요."
assert health.json() == {"status": "ok"}, f"GET /healthz 의 본문이 {health.json()}입니다. 제공한 system_router 를 그대로 붙이세요."

# 모델 목록 주소: 같은 파드의 vLLM 에 물어 서빙 중인 모델 이름을 돌려줍니다
models = httpx.get(f"{SERVING_URL}/v1/models", timeout=REQUEST_TIMEOUT)
assert models.status_code == 200, f"GET /v1/models 가 {models.status_code}입니다. 404 면 system_router 를 붙였는지, 500 이면 터미널 1의 vLLM 서버가 떠 있는지 확인하세요."
assert SERVED_NAME in models.json()["models"], f"모델 목록이 {models.json()['models']}입니다. 터미널 1의 vLLM 서버가 --served-model-name {SERVED_NAME} 으로 떠 있는지 확인하세요."
print("✅ 통과!")

## 2. 요청 양식

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 쇼핑 질문의 요청 양식을 pydantic 모델로 정합니다. 빈 질문, 상담 창의 입력 한도(300자)를 넘는 질문, 형식이 틀린 주문번호는 vLLM에 보내기 전에 422로 돌려보냅니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/schemas/chat.py</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">question</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">str</code>이고 기본값이 없는 필수 항목입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>로 2자 이상 300자 이하로 제한합니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">think</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">bool</code>이고 기본값은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">False</code>입니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">order_id</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">str | None</code>이고 기본값은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>입니다(빠져도 됩니다). 값이 있으면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Field</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pattern</code>으로 정규식 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">^ORD-\d{6}&#36;</code>(ORD- 뒤에 숫자 6자리)에 맞게 제한합니다.</li>
</ul>
<p style="margin:6px 0;">항목은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">question</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">order_id</code> 순서로 둡니다.</p>
<p style="margin:6px 0;">요청 양식은 그 양식을 받는 주소가 생겨야 서버에서 확인할 수 있습니다. 그래서 이 문항의 자가채점 셀은 3번 끝에 있고, 3번까지 채운 뒤 실행합니다. 이 셀은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/openapi.json</code>에 실린 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code> 설명(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/docs</code>의 Schemas에 보이는 설명)과, 양식에 맞는 요청과 어긋난 요청의 상태 코드를 확인합니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 01의 4-1(상담 요청과 응답의 스키마)을 참고합니다.
세부구현
1) 항목마다 이름: 자료형 = Field(...) 모양으로 씁니다.
2) 글자 수 제한(min_length, max_length), 기본값(default), 정규식(pattern)은 Field의 키워드 인자로 줍니다.
3) 빠져도 되는 항목은 자료형에 None을 함께 허용하고 기본값을 None으로 둡니다.
```

</details>

## 3. 쇼핑 상담 주소

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/shop</code> 아래에 주소 두 개를 만들어, 주문 내역 화면은 답 전체를 한 번에 받고 상담 채팅 창은 조각이 생기는 대로 받게 합니다. 주문번호가 있으면 사용자 메시지 앞에 붙여 모델이 어느 주문의 질문인지 알게 합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/routers/shop_router.py</code>)</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">shop_router</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">APIRouter</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">prefix</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;/v1/shop&quot;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tags</code>는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[&quot;쇼핑 상담&quot;]</code>입니다.</li>
<li style="margin:3px 0;">(2) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">make_user_message(request)</code>:</strong> 모델에 보낼 사용자 메시지(str)를 돌려줍니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request.order_id</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>이면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request.question</code> 그대로이고, 아니면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[주문번호 &lt;order_id&gt;] &lt;question&gt;</code> 모양(대괄호 뒤에 빈칸 하나)입니다. 자가채점은 같은 질문을 주문번호 없이 한 번, 붙여서 한 번 보내고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage.prompt_tokens</code>의 차이가 질문 앞에 붙는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">[주문번호 ORD-123456] </code>의 토큰 수(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ORDER_PREFIX_TOKENS</code>, 11)와 같은지 봅니다. 모양이 조금만 달라도 토큰 수가 달라집니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">shop_chat</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat</code> 주소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 함수입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">요청 본문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code>, 응답 양식은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">response_model=ChatResponse</code>입니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">make_user_message</code>의 메시지와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ask_vllm</code>을 부르고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">to_chat_response</code>로 바꿔 돌려줍니다.</li>
<li style="margin:3px 0;">응답 키는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">finish_reason</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">model</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>이고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">False</code>면 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">None</code>입니다. 2번 양식에 어긋난 요청은 422입니다.</li>
</ul>
</li>
<li style="margin:3px 0;">(4) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">shop_chat_stream</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">POST /v1/shop/chat/stream</code> 주소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">async def</code> 함수입니다. 요청 본문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ShopChatRequest</code>입니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">3번처럼 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">make_user_message(request)</code>로 만든 메시지와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">request.think</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">open_vllm_stream</code>에 넘겨 스트림을 먼저 열고, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">relay_stream</code>의 조각을 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">StreamingResponse</code>에 담습니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">media_type</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&quot;text/event-stream&quot;</code>입니다.</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data:</code> 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">type</code>은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code> 순서이고 마지막 줄은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data: [DONE]</code>입니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">reasoning</code> 조각은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">think</code>가 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">True</code>일 때만 오고 모두 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">answer</code> 조각보다 앞에 옵니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">usage</code>는 맨 끝에 하나만 옵니다.</li>
</ul>
</li>
</ul>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 01의 3절(라우팅), 4-2(vLLM에 요청하는 코드), 6-1(SSE와 relay_stream)을 참고합니다.
세부구현
1) 주소 묶음은 APIRouter에 prefix와 tags 키워드 인자를 줍니다.
2) make_user_message는 if 문으로 order_id가 None인지 먼저 보고, 아니면 f-string으로 메시지를 만듭니다.
3) 주소 함수는 shop_router의 post 데코레이터를 붙인 async def이고, 답 전체 주소의 데코레이터에는 response_model도 줍니다.
4) ask_vllm, open_vllm_stream은 코루틴이므로 await로 결과를 받습니다.
5) 스트리밍 주소는 relay_stream이 돌려주는 비동기 제너레이터를 StreamingResponse에 넘깁니다.
```

</details>

In [ ]:
# [자가채점] /v1/shop/chat 의 등록과 응답 양식, 주문번호, 양식 오류(422)를 확인합니다
openapi = httpx.get(f"{SERVING_URL}/openapi.json", timeout=REQUEST_TIMEOUT).json()
assert "/v1/shop/chat" in openapi["paths"], f"서버 주소 목록에 /v1/shop/chat 이 없습니다(있는 주소: {list(openapi['paths'])}). routers/shop_router.py 의 TODO (1) prefix, TODO (3) 데코레이터, main.py 의 TODO (2) include_router 를 확인하세요."

# (3) 응답 양식과 요청 본문 양식
chat_operation = openapi["paths"]["/v1/shop/chat"]["post"]
response_ref = chat_operation["responses"]["200"]["content"]["application/json"]["schema"].get("$ref", "")
assert response_ref.endswith("/ChatResponse"), "POST /v1/shop/chat 의 응답 양식이 ChatResponse 가 아닙니다. 데코레이터에 response_model=ChatResponse 를 주세요(TODO (3))."
assert "requestBody" in chat_operation, "POST /v1/shop/chat 에 요청 본문이 없습니다. 함수 인자로 ShopChatRequest 를 받으세요(TODO (3))."
request_ref = chat_operation["requestBody"]["content"]["application/json"]["schema"].get("$ref", "")
assert request_ref.endswith("/ShopChatRequest"), "POST /v1/shop/chat 의 요청 본문 양식이 ShopChatRequest 가 아닙니다. 함수 인자의 자료형을 ShopChatRequest 로 두세요(TODO (3))."

# (3) 답 전체 응답: 생각을 끈 요청
plain = httpx.post(f"{SERVING_URL}/v1/shop/chat", json={"question": sample_question}, timeout=REQUEST_TIMEOUT)
assert plain.status_code == 200, f"POST /v1/shop/chat 이 {plain.status_code}입니다. 500 이면 터미널 2의 마지막 오류 줄을 보세요(NotImplementedError 면 그 TODO 가 비어 있습니다). 본문 앞부분: {plain.text[:200]}"
plain_body = plain.json()
assert set(plain_body) == {"answer", "reasoning", "finish_reason", "model", "usage"}, f"응답 키가 {list(plain_body)}입니다. to_chat_response 의 결과를 그대로 돌려주세요(TODO (3))."
assert isinstance(plain_body["answer"], str) and plain_body["reasoning"] is None, "think 를 주지 않은 요청은 answer 가 문자열이고 reasoning 이 None 이어야 합니다. ask_vllm 에 request.think 를 넘겼는지 확인하세요(TODO (3))."
assert plain_body["model"] == SERVED_NAME and plain_body["finish_reason"] in ["stop", "length"], f"model 이 {plain_body['model']!r}, finish_reason 이 {plain_body['finish_reason']!r}입니다. to_chat_response 로 바꾼 값을 돌려주세요."
plain_usage = plain_body["usage"]
assert plain_usage["total_tokens"] == plain_usage["prompt_tokens"] + plain_usage["completion_tokens"], f"usage 가 {plain_usage}입니다. 입력과 출력 토큰 수의 합이 total_tokens 여야 합니다."

# (3) 생각을 켠 요청: reasoning 이 채워져 옵니다(생각이 길어 1분 가까이 걸릴 수 있습니다)
thinking = httpx.post(f"{SERVING_URL}/v1/shop/chat", json={"question": sample_question, "think": True}, timeout=REQUEST_TIMEOUT)
assert thinking.status_code == 200, f"think=True 요청이 {thinking.status_code}입니다. 본문 앞부분: {thinking.text[:200]}"
assert thinking.json()["reasoning"] is not None, "think=True 인데 reasoning 이 None 입니다. ask_vllm 에 request.think 를 넘겼는지 확인하세요(TODO (3))."

# (2) 주문번호: 같은 질문에 주문번호를 주면 입력 토큰 수가 "[주문번호 ORD-123456] " 의 토큰 수만큼 늘어납니다
ordered = httpx.post(f"{SERVING_URL}/v1/shop/chat", json={"question": sample_question, "order_id": "ORD-123456"}, timeout=REQUEST_TIMEOUT)
assert ordered.status_code == 200, f"주문번호 ORD-123456 을 준 요청이 {ordered.status_code}입니다. 맞는 형식의 주문번호는 받아야 합니다(schemas/chat.py 의 TODO (3), routers/shop_router.py 의 TODO (2))."
added_tokens = ordered.json()["usage"]["prompt_tokens"] - plain_usage["prompt_tokens"]
assert added_tokens > 0, f"주문번호를 준 요청의 입력 토큰 수가 주지 않은 요청보다 크지 않습니다(차이 {added_tokens}). make_user_message 가 주문번호를 질문 앞에 붙이고, shop_chat 이 그 메시지를 ask_vllm 에 넘기는지 확인하세요(TODO (2), (3))."
assert added_tokens == ORDER_PREFIX_TOKENS, f"주문번호를 붙여 늘어난 입력 토큰 수가 {added_tokens}입니다. \"[주문번호 ORD-123456] \" 을 질문 앞에 붙이면 {ORDER_PREFIX_TOKENS}이어야 합니다. 대괄호, '주문번호' 글자, 대괄호 뒤 빈칸 하나, 질문보다 앞이라는 순서를 요구사항 (2)의 모양과 하나씩 맞춰 보세요(TODO (2))."

# 2번 양식에 어긋난 요청(질문이 없음)은 vLLM 에 보내기 전에 422 로 돌아옵니다
no_question = httpx.post(f"{SERVING_URL}/v1/shop/chat", json={"think": True}, timeout=REQUEST_TIMEOUT)
assert no_question.status_code == 422, f"질문이 없는 요청이 {no_question.status_code}입니다. 422 여야 합니다. 함수 인자의 자료형이 ShopChatRequest 인지 확인하세요(TODO (3))."
print("✅ 통과!")

In [ ]:
# [자가채점] /v1/shop/chat/stream 의 응답 형식과 SSE 이벤트 순서를 확인합니다
def read_stream(payload):
    """스트리밍 주소에 요청을 보내고 (상태 코드, content-type, 이벤트 리스트, 마지막 data 줄)을 돌려줍니다."""
    events = []
    last_line = ""
    with httpx.stream("POST", f"{SERVING_URL}/v1/shop/chat/stream", json=payload, timeout=REQUEST_TIMEOUT) as response:
        for line in response.iter_lines():
            # SSE 의 한 이벤트는 "data: " 로 시작하는 줄 하나입니다(빈 줄은 이벤트 구분)
            if not line.startswith("data: "):
                continue
            last_line = line[len("data: "):]
            if last_line != "[DONE]":
                events.append(json.loads(last_line))
    return response.status_code, response.headers.get("content-type", ""), events, last_line


# 생각을 끈 요청: answer 조각들, usage 하나, [DONE]
status, content_type, events, last_line = read_stream({"question": sample_question})
assert status == 200, f"POST /v1/shop/chat/stream 이 {status}입니다. 404 면 TODO (4)의 데코레이터 주소를, 500 이면 터미널 2의 마지막 오류 줄을 확인하세요."
assert content_type.startswith("text/event-stream"), f"content-type 이 {content_type!r}입니다. StreamingResponse 에 media_type=\"text/event-stream\" 을 주세요(TODO (4))."
assert last_line == "[DONE]", f"마지막 data 줄이 {last_line!r}입니다. relay_stream 의 조각을 그대로 보내세요(TODO (4))."
event_types = []
for event in events:
    event_types.append(event["type"])
assert "answer" in event_types and "reasoning" not in event_types, f"생각을 끈 요청의 이벤트 type 이 {sorted(set(event_types))}입니다. answer 만 있어야 합니다. open_vllm_stream 에 request.think 를 넘겼는지 확인하세요."
assert event_types[-1] == "usage" and event_types.count("usage") == 1, f"usage 이벤트가 {event_types.count('usage')}개이고 마지막 type 이 {event_types[-1]!r}입니다. usage 하나가 [DONE] 바로 앞에 와야 합니다."
assert set(events[-1]) == {"type", "prompt_tokens", "cached_tokens", "completion_tokens", "total_tokens"}, f"usage 이벤트의 키가 {list(events[-1])}입니다."
plain_stream_tokens = events[-1]["prompt_tokens"]

# 주문번호를 준 요청: usage 의 입력 토큰 수가 "[주문번호 ORD-123456] " 의 토큰 수만큼 늘어납니다
status, content_type, events, last_line = read_stream({"question": sample_question, "order_id": "ORD-123456"})
assert status == 200 and last_line == "[DONE]", f"주문번호를 준 스트리밍 요청이 {status}, 마지막 줄 {last_line!r}입니다."
added_stream_tokens = events[-1]["prompt_tokens"] - plain_stream_tokens
assert added_stream_tokens == ORDER_PREFIX_TOKENS, f"스트리밍 주소에서 주문번호를 붙여 늘어난 입력 토큰 수가 {added_stream_tokens}입니다. {ORDER_PREFIX_TOKENS}이어야 합니다. make_user_message(request) 로 만든 메시지를 open_vllm_stream 에 넘기세요(TODO (4))."

# 생각을 켠 요청: reasoning 조각이 모두 answer 조각보다 앞에 옵니다
status, content_type, events, last_line = read_stream({"question": sample_question, "think": True})
assert status == 200 and last_line == "[DONE]", f"think=True 스트리밍 요청이 {status}, 마지막 줄 {last_line!r}입니다."
event_types = []
for event in events:
    event_types.append(event["type"])
assert "reasoning" in event_types, "think=True 인데 reasoning 이벤트가 없습니다. open_vllm_stream 에 request.think 를 넘겼는지 확인하세요(TODO (4))."
answer_started = False
for event_type in event_types:
    if event_type == "answer":
        answer_started = True
    assert not (answer_started and event_type == "reasoning"), "answer 조각 뒤에 reasoning 조각이 왔습니다. 제공한 relay_stream 을 그대로 쓰세요."
assert event_types[-1] == "usage", f"마지막 이벤트 type 이 {event_types[-1]!r}입니다. usage 가 [DONE] 바로 앞에 와야 합니다."
print("✅ 통과!")

In [ ]:
# [자가채점] 2번 요청 양식: 3번의 /v1/shop/chat 주소로 ShopChatRequest 의 세 항목을 확인합니다
openapi = httpx.get(f"{SERVING_URL}/openapi.json", timeout=REQUEST_TIMEOUT).json()
# 서버 문서의 양식 목록(주소에 양식이 하나도 없으면 components 항목 자체가 없습니다)
document_schemas = openapi.get("components", {}).get("schemas", {})
assert "ShopChatRequest" in document_schemas, "서버 문서에 ShopChatRequest 가 없습니다. 3번의 shop_chat 이 ShopChatRequest 를 요청 본문으로 받게 한 뒤 실행하세요."
request_schema = document_schemas["ShopChatRequest"]
properties = request_schema.get("properties", {})
assert list(properties) == ["question", "think", "order_id"], f"ShopChatRequest 의 항목이 {list(properties)}입니다. schemas/chat.py 에 question, think, order_id 순서로 세 항목을 두세요."

# (1) question: 필수, 문자열, 2자 이상 300자 이하
question_schema = properties["question"]
assert question_schema.get("type") == "string", f"question 의 설명이 {question_schema}입니다. 자료형을 str 하나로 두세요(TODO (1))."
assert question_schema.get("minLength") == 2 and question_schema.get("maxLength") == 300, f"question 의 글자 수 제한이 {question_schema.get('minLength')}~{question_schema.get('maxLength')}입니다. Field 의 min_length=2, max_length=300 을 주세요(TODO (1))."
assert request_schema.get("required", []) == ["question"], f"필수 항목이 {request_schema.get('required', [])}입니다. question 에만 기본값을 두지 마세요(think, order_id 에는 기본값이 있습니다)."

# (2) think: 참거짓, 기본값 False
think_schema = properties["think"]
assert think_schema.get("type") == "boolean" and think_schema.get("default") is False, f"think 의 설명이 {think_schema}입니다. 자료형 bool, 기본값 False 로 두세요(TODO (2))."

# (3) order_id: 문자열 또는 None, 기본값 None(빠져도 되는 항목), 문자열이면 정규식 ^ORD-\d{6}$
order_schema = properties["order_id"]
assert "order_id" not in request_schema.get("required", []), "order_id 가 필수 항목입니다. 기본값을 None 으로 두어 빠져도 되게 하세요(TODO (3))."
order_types = []
order_patterns = []
for option in order_schema.get("anyOf", [order_schema]):
    order_types.append(option.get("type"))
    if "pattern" in option:
        order_patterns.append(option["pattern"])
assert sorted(order_types) == ["null", "string"], f"order_id 가 받는 자료형이 {order_types}입니다. 자료형을 str | None 으로 두세요(TODO (3))."
assert order_patterns == [r"^ORD-\d{6}$"], f"order_id 의 정규식이 {order_patterns}입니다. Field 의 pattern 에 요구사항의 정규식(ORD- 뒤에 숫자 6자리)을 그대로 주세요(TODO (3))."

# 양식의 경계에 맞는 질문(2자, 300자)은 200 입니다
for question in [sample_question[:2], (sample_question * 10)[:300]]:
    response = httpx.post(f"{SERVING_URL}/v1/shop/chat", json={"question": question}, timeout=REQUEST_TIMEOUT)
    assert response.status_code == 200, f"{len(question)}자 질문이 {response.status_code}입니다. question 은 2자 이상 300자 이하를 받아야 합니다(TODO (1))."

# 양식에 어긋난 요청은 vLLM 에 보내기 전에 422 로 돌아옵니다
invalid_requests = [
    {"question": "환"},                                       # 2자보다 짧은 질문
    {"question": "가" * 301},                                 # 300자보다 긴 질문
    {"think": True},                                          # 질문이 없음
    {"question": sample_question, "order_id": "ORD-12345"},  # 숫자가 5자리
    {"question": sample_question, "order_id": "ord-123456"},  # 소문자 ord
]
for payload in invalid_requests:
    response = httpx.post(f"{SERVING_URL}/v1/shop/chat", json=payload, timeout=REQUEST_TIMEOUT)
    assert response.status_code == 422, f"{str(payload)[:60]} 요청이 {response.status_code}입니다. 422 여야 합니다. schemas/chat.py 의 ShopChatRequest 를 확인하세요."
print("✅ 통과!")

## 마무리

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">과제 02에서는 과제 02 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api</code>(과제 01의 완성 코드가 들어 있습니다)에 서비스 키, 요청 한도, 오류 양식을 더합니다. 교안 02로 넘어가도 터미널 1의 vLLM 서버는 그대로 둡니다.</p>
</div>

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">오늘 실습을 여기서 마칠 때</summary>

<div class="day77-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 2</strong>(FastAPI 서버)와 <strong style="font-size:1.13em;color:#1f3a5f;">RunPod 터미널 1</strong>(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">qwen3.5-4b</code> vLLM 서버)에서 차례로 <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Ctrl+C</code> 키</strong>를 누릅니다. 명령을 입력할 수 있게 되면 터미널 1에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">nvidia-smi</code>를 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 메모리 사용량(<code style="color:inherit;background:#f3f6fa;font-size:0.95em;">Memory-Usage</code>)이 서버를 띄우기 전 수준으로 줄었으면</strong> vLLM 서버가 꺼진 것입니다.</p>
<p style="margin:6px 0;">오늘 실습을 마치면 RunPod 콘솔에서 파드를 종료(Terminate)합니다. 정지(Stop)해도 다음에 같은 GPU로 다시 켜지지 않는 경우가 많습니다. 남길 파일(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output</code> 등)이 있으면 종료 전에 내 PC로 받아 둡니다.</p>
</div>

</details>